# умножение на cos + инвертирование

In [3]:
from scipy.io import wavfile
import numpy as np
from scipy import signal
from scipy.signal import kaiser_beta
from pystoi.stoi import stoi
# from pesq import pesq
import pandas as pd
import matplotlib.pyplot as plt
import subprocess, textwrap
import os
import hashlib

#### построение фильтра с нужными таймингами и подавлением


In [4]:
def design_fir_lowpass(length_sec: float, cutoff_hz: float, fs: float, atten_db: float) -> np.ndarray:
    '''
    length_sec - длина фильтра (в секундах)
    cutoff_hz - чатсота среза
    fs - частота дискретизации
    atten_db - в Дб насколько глушим мусор
    returning: taps - коэффициенты FIR-фильтра (его ИХ)
    '''
    if length_sec <= 0.0:
        # длина 1 => просто "пропустить как есть"
        return np.array([1.0], dtype=float)

    max_len = 2047
    taps_len = int(2 * round(fs * length_sec) + 1)
    taps_len = min(taps_len, max_len)
    if taps_len < 3:
        taps_len = 3#вычисление длины фильтра

    beta = kaiser_beta(atten_db)
    taps = signal.firwin(taps_len, cutoff=cutoff_hz, window=("kaiser", beta), fs=fs)
    return taps.astype(float)

#### умножение сигнала на cos (основная ф-я)

In [5]:
def inverter_offline(x: np.ndarray, fs: float, freq_prefilter: float, freq_shift: float, freq_postfilter: float,  length_sec: float = 0.0024, atten_db: float = 60.0) -> np.ndarray:
    """
    x - отсчеты сигнала
    fs - частота дискретизации
    freq_prefilter - чатсота префильтра чтобы сделать полосу достаточно узкой (чтобы не доставала до 0 и до fs/2), чтобы не было наложения
    freq_shift - несущая
    freq_postfilter - частота постфильтра чтобы обрезать вторую копию и оставить нужную полосу
    length_sec - контролим длину фильтра
    atten_db - контролим как сильно прибиваем мусор
    prefilter -> умножение на cos(2π f_shift t) -> postfilter
    """
    n = np.arange(len(x), dtype=float)#n — индексы отсчётов
    omega = 2.0 * np.pi * freq_shift / fs
    carrier = np.cos(omega * n)
    
    # prefilter
    taps_pre = design_fir_lowpass(length_sec, freq_prefilter, fs, atten_db)
    x_pre = signal.lfilter(taps_pre, [1.0], x)

    mixed = x_pre * carrier

    # postfilter
    taps_post = design_fir_lowpass(length_sec, freq_postfilter, fs, atten_db)
    y = signal.lfilter(taps_post, [1.0], mixed)

    return y

### main (simple/split mode)

In [6]:
def to_mono_float(data: np.ndarray) -> np.ndarray:
    if data.dtype.kind in ("i", "u"):
        data = data.astype(np.float32) / np.iinfo(data.dtype).max
    else:
        data = data.astype(np.float32)

    if data.ndim > 1:
        data = data[:, 0]
    return data

def simple_process(x: np.ndarray, fs: int, freq_hi: float) -> np.ndarray:
    return inverter_offline(x, fs, freq_prefilter=freq_hi, freq_shift=freq_hi, freq_postfilter=freq_hi)

def split_process(x: np.ndarray, fs: int, freq_lo: float, freq_hi: float) -> np.ndarray:
    y1 = inverter_offline(x, fs, freq_prefilter=freq_lo, freq_shift=freq_lo, freq_postfilter=freq_lo)

    y2 = inverter_offline(x, fs, freq_prefilter=freq_hi, freq_shift=(freq_lo + freq_hi), freq_postfilter=freq_hi)

    return y1 + y2

def write_wav(path, fs, sig):
    sig = sig.astype(np.float32)
    m = np.max(np.abs(sig)) + 1e-12
    sig = sig / m  # нормализация по пику, чтобы не клипповало
    wavfile.write(path, fs, (sig * 32767).astype(np.int16))

инверсия знака

In [8]:
KEY_PATH = "./key.bin"

def load_key_bytes(path):
    with open(path, "rb") as f:
        return f.read()

def _seed_from_key(key_bytes, salt):
    h = hashlib.sha256()
    h.update(key_bytes)
    h.update(str(salt).encode("utf-8"))
    return int.from_bytes(h.digest()[:8], "big", signed=False)

def sign_mask_vector(size, key_bytes, salt=""):
    rng = np.random.default_rng(_seed_from_key(key_bytes, salt))
    return rng.choice(np.array([-1.0, 1.0]), size=size).astype(np.float64)

def sign_mask_blocks(n_samples, block_size, key_bytes, salt=""):
    rng = np.random.default_rng(_seed_from_key(key_bytes, salt))
    n_blocks = (n_samples + block_size - 1) // block_size
    signs = rng.choice(np.array([-1.0, 1.0]), size=n_blocks).astype(np.float64)
    return np.repeat(signs, block_size)[:n_samples]

key_bytes = load_key_bytes(KEY_PATH)

инверсия для симпл и сплит модов

In [9]:
def simple_sign_process(x, fs, freq_hi, key_bytes, block_size=1024):
    y = simple_process(x, fs, freq_hi=freq_hi)
    mask = sign_mask_blocks(len(y), block_size, key_bytes, salt=f"simple_sign_{len(y)}_{fs}_{freq_hi}")
    return y * mask

def split_sign_process(x, fs, freq_lo, freq_hi, key_bytes, block_size=1024):
    y = split_process(x, fs, freq_lo=freq_lo, freq_hi=freq_hi)
    mask = sign_mask_blocks(len(y), block_size, key_bytes, salt=f"split_sign_{len(y)}_{fs}_{freq_lo}_{freq_hi}")
    return y * mask

In [10]:
DATASET_DIR = "../../dataset/test_sounds"

OUT_ENC_SIMPLE_DIR = "./assets/encrypted/simple"
OUT_DEC_SIMPLE_DIR = "./assets/decrypted/simple"

OUT_ENC_SPLIT_DIR = "./assets/encrypted/split"
OUT_DEC_SPLIT_DIR = "./assets/decrypted/split"

OUT_ENC_SIMPLE_SIGN_DIR = "./assets/encrypted/simple_sign"
OUT_DEC_SIMPLE_SIGN_DIR = "./assets/decrypted/simple_sign"

OUT_ENC_SPLIT_SIGN_DIR = "./assets/encrypted/split_sign"
OUT_DEC_SPLIT_SIGN_DIR = "./assets/decrypted/split_sign"

os.makedirs(OUT_ENC_SIMPLE_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SIMPLE_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SPLIT_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SPLIT_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SIMPLE_SIGN_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SIMPLE_SIGN_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SPLIT_SIGN_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SPLIT_SIGN_DIR, exist_ok=True)

N_FILES = 23

freq_hi_simple = 2632.0
freq_lo_split = 500.0
freq_hi_split = 2500.0

for i in range(1, N_FILES + 1):
    in_path = os.path.join(DATASET_DIR, f"test_sound{i:1d}.wav")

    fs, data = wavfile.read(in_path)
    x = to_mono_float(data)

    simple_enc = simple_process(x, fs, freq_hi=freq_hi_simple)
    simple_dec = simple_process(simple_enc, fs, freq_hi=freq_hi_simple)

    write_wav(os.path.join(OUT_ENC_SIMPLE_DIR, f"test_sound{i:1d}_simple_enc.wav"), fs, simple_enc)
    write_wav(os.path.join(OUT_DEC_SIMPLE_DIR, f"test_sound{i:1d}_simple_dec.wav"), fs, simple_dec)

    if freq_lo_split >= freq_hi_split:
        raise ValueError("freq_lo must be < freq_hi")
    if fs < 2.0 * freq_hi_split:
        raise ValueError("sample rate must be at least 2*freq_hi")

    split_enc = split_process(x, fs, freq_lo=freq_lo_split, freq_hi=freq_hi_split)
    split_dec = split_process(split_enc, fs, freq_lo=freq_lo_split, freq_hi=freq_hi_split)

    write_wav(os.path.join(OUT_ENC_SPLIT_DIR, f"test_sound{i:1d}_split_enc.wav"), fs, split_enc)
    write_wav(os.path.join(OUT_DEC_SPLIT_DIR, f"test_sound{i:1d}_split_dec.wav"), fs, split_dec)

    simple_sign_enc = simple_sign_process(x, fs, freq_hi=freq_hi_simple, key_bytes=key_bytes)
    simple_sign_dec_mid = simple_sign_process(simple_sign_enc, fs, freq_hi=freq_hi_simple, key_bytes=key_bytes)
    simple_sign_dec = simple_process(simple_sign_dec_mid, fs, freq_hi=freq_hi_simple)

    write_wav(os.path.join(OUT_ENC_SIMPLE_SIGN_DIR, f"test_sound{i:1d}_simple_sign_enc.wav"), fs, simple_sign_enc)
    write_wav(os.path.join(OUT_DEC_SIMPLE_SIGN_DIR, f"test_sound{i:1d}_simple_sign_dec.wav"), fs, simple_sign_dec)

    split_sign_enc = split_sign_process(x, fs, freq_lo=freq_lo_split, freq_hi=freq_hi_split, key_bytes=key_bytes)
    split_sign_dec_mid = split_sign_process(split_sign_enc, fs, freq_lo=freq_lo_split, freq_hi=freq_hi_split, key_bytes=key_bytes)
    split_sign_dec = split_process(split_sign_dec_mid, fs, freq_lo=freq_lo_split, freq_hi=freq_hi_split)

    write_wav(os.path.join(OUT_ENC_SPLIT_SIGN_DIR, f"test_sound{i:1d}_split_sign_enc.wav"), fs, split_sign_enc)
    write_wav(os.path.join(OUT_DEC_SPLIT_SIGN_DIR, f"test_sound{i:1d}_split_sign_dec.wav"), fs, split_sign_dec)